In [3]:
import os
import json

model_name_to_path = {
    "Mistral-Small-24B-Instruct-2501": "mistralai/Mistral-Small-24B-Instruct-2501",
    "Meta-Llama-3.1-8B-Instruct": "meta-llama/Llama-3.1-8B-Instruct",
    "Mistral-7B-Instruct-v0.2": "mistralai/Mistral-7B-Instruct-v0.2",
    "Qwen2.5-32B-Instruct": "Qwen/Qwen2.5-32B-Instruct"
}

selected_tasks = ['gov_report', 'paper_assistant']

model_to_unstable_heads = {
    "Mistral-Small-24B-Instruct-2501": 80,
    "Meta-Llama-3.1-8B-Instruct": 64,
    "Mistral-7B-Instruct-v0.2": 64,
    "Qwen2.5-32B-Instruct": 128
}

In [4]:
base_path = 'Data-Sorted/analysis_outputs'

processed_data = {}

for topk_name in os.listdir(base_path):
    k = int(topk_name.split('-')[-1])
    for model_name in os.listdir(os.path.join(base_path, topk_name)):
        num_unstable_heads = model_to_unstable_heads.get(model_name)
        for task_name in selected_tasks:
            data_file = os.path.join(
                base_path, topk_name, model_name, task_name, f'consensus_M{num_unstable_heads}.json'
            )
            data = json.load(open(data_file, 'r'))
            unstable_heads = data['topM']
            # ALL THESE MODELS HAVE 8 KV HEADS
            H = 8
            unstable_heads = [(h // H, h % H) for h in unstable_heads]

            model_path = model_name_to_path[model_name]

            if model_path not in processed_data:
                processed_data[model_path] = {}

            head_key = f'unstable-{num_unstable_heads}-profile-{task_name}-topk-{k}'

            processed_data[model_path][head_key] = unstable_heads

output_file = 'model_unstable_heads_data.json'
with open(output_file, 'w') as f:
    json.dump(processed_data, f, indent=4)